#### Imports & Setup

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
from pathlib import Path
import sys

# Project root detection
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR

print(f"Project Root: {PROJECT_ROOT}")

# Create directories
dirs = [
    PROJECT_ROOT / "data" / "raw",
    PROJECT_ROOT / "data" / "processed",
    PROJECT_ROOT / "src",
    PROJECT_ROOT / "app",
    PROJECT_ROOT / "notebooks"
]

for d in dirs:
    d.mkdir(parents=True, exist_ok=True)

print("Project directories created successfully!")

Project Root: C:\Users\tohiba\Desktop\traceability-responsible-sourcing-map
Project directories created successfully!


#### Define Data Schema (Core Columns)

In [2]:
columns = {
    'lot_id': 'str',
    'supplier_name': 'str',
    'washing_station': 'str',
    'farm_group': 'str',
    'region': 'str',
    'district': 'str',
    
    # Geolocation
    'latitude': 'float',
    'longitude': 'float',
    'altitude_m': 'float',
    
    'processing_method': 'str',
    'harvest_year': 'int',
    'arrival_date': 'datetime64[ns]',
    
    # Traceability
    'traceability_level': 'str',
    
    # Quality
    'sca_score': 'float',
    'grade_ecx': 'str',
    'defects_per_300g': 'int',
    
    # Responsible Sourcing & EUDR
    'sustainability_cert': 'str',
    'deforestation_risk': 'str',
    'eudr_compliant': 'bool',
    'farmer_premium_usd': 'float',
    'last_audit_date': 'datetime64[ns]',
    'compliance_status': 'str',
    
    # Commercial
    'quantity_bags': 'int',
    'available_quantity_kg': 'float',
    'price_per_kg_usd': 'float',
    
    'status': 'str',
    'last_updated': 'datetime64[ns]'
}

print("Professional Traceability Schema Defined (EUDR Ready)")
print(f"Total Columns: {len(columns)}")

Professional Traceability Schema Defined (EUDR Ready)
Total Columns: 27


#### Create Mock Data (Realistic Ethiopian Lots)

In [7]:
np.random.seed(42)
n = 50

# ==================== ACCURATE REAL GPS RANGES ====================
region_data = {
    'Yirgacheffe': {
        'lat_range': (5.95, 6.35),      # Gedeo Zone
        'lon_range': (38.10, 38.45),
        'alt_range': (1750, 2200)
    },
    'Guji': {
        'lat_range': (5.50, 6.20),      # Guji Zone
        'lon_range': (38.80, 39.45),
        'alt_range': (1650, 2250)
    },
    'Sidama': {
        'lat_range': (6.45, 7.10),      # Sidama Region
        'lon_range': (38.10, 38.75),
        'alt_range': (1550, 2100)
    },
    'Limmu': {
        'lat_range': (7.85, 8.35),      # Limmu Kossa area
        'lon_range': (36.45, 37.15),
        'alt_range': (1400, 1950)
    }
}

regions = list(region_data.keys())

data = {
    'lot_id': [], 'supplier_name': [], 'washing_station': [], 'farm_group': [],
    'region': [], 'district': [], 'latitude': [], 'longitude': [], 'altitude_m': [],
    'processing_method': [], 'harvest_year': [], 'arrival_date': [],
    'traceability_level': [], 'sca_score': [], 'grade_ecx': [], 'defects_per_300g': [],
    'sustainability_cert': [], 'deforestation_risk': [], 'eudr_compliant': [],
    'farmer_premium_usd': [], 'last_audit_date': [], 'compliance_status': [],
    'quantity_bags': [], 'price_per_kg_usd': [], 'status': []
}

for i in range(n):
    region = np.random.choice(regions)
    info = region_data[region]
    
    data['lot_id'].append(f"ETH-2025-{region[:3].upper()}-{str(i+1).zfill(3)}")
    data['supplier_name'].append(np.random.choice(['Daye Bensa', 'Hambela Coffee', 'Misty Valley', 'Guji Highland', 'Sidama Crown', 'Red Cherry Export']))
    data['washing_station'].append(np.random.choice(['Hambela', 'Daye Bensa', 'Gedeb', 'Uraga', 'Bensa', 'Shakiso']))
    data['farm_group'].append(np.random.choice(['Yirga Chefe Coop', 'Guji Organic', 'Sidama Union', 'Limmu Coop']))
    data['region'].append(region)
    data['district'].append(np.random.choice(['Yirga Chefe', 'Gedeb', 'Shakiso', 'Hambela', 'Bensa', 'Uraga']))
    
    # Accurate GPS
    data['latitude'].append(np.round(np.random.uniform(*info['lat_range']), 6))
    data['longitude'].append(np.round(np.random.uniform(*info['lon_range']), 6))
    data['altitude_m'].append(np.random.randint(*info['alt_range']))
    
    data['processing_method'].append(np.random.choice(['Washed', 'Natural', 'Honey'], p=[0.55, 0.35, 0.10]))
    data['harvest_year'].append(2025)
    data['arrival_date'].append(pd.Timestamp('2026-02-01') + pd.Timedelta(days=i*3))
    
    data['traceability_level'].append(np.random.choice(['Full Farm', 'Washing Station', 'Exporter'], p=[0.45, 0.40, 0.15]))
    data['sca_score'].append(np.round(np.random.uniform(82.0, 92.8), 1))
    data['grade_ecx'].append(np.random.choice(['Grade 1', 'Grade 2'], p=[0.70, 0.30]))
    data['defects_per_300g'].append(np.random.randint(0, 12))
    
    data['sustainability_cert'].append(np.random.choice(['Organic', 'Volcafe Verified', 'Rainforest Alliance', 'None']))
    data['deforestation_risk'].append(np.random.choice(['Low', 'Medium', 'High'], p=[0.78, 0.18, 0.04]))
    data['eudr_compliant'].append(np.random.choice([True, False], p=[0.88, 0.12]))
    data['farmer_premium_usd'].append(np.round(np.random.uniform(0.25, 0.95), 2))
    data['last_audit_date'].append(pd.Timestamp('2025-09-01') + pd.Timedelta(days=i*12))
    data['compliance_status'].append(np.random.choice(['Compliant', 'In Progress', 'At Risk'], p=[0.82, 0.14, 0.04]))
    
    data['quantity_bags'].append(np.random.randint(15, 180))
    data['price_per_kg_usd'].append(np.round(np.random.uniform(5.8, 9.8), 2))
    data['status'].append('Available')

df = pd.DataFrame(data)
df['available_quantity_kg'] = df['quantity_bags'] * 60.0
df['last_updated'] = datetime.now()

print("Regional GPS Data Generated")
print(f"Total Lots: {len(df)}")
print("\nRegion Distribution:")
print(df['region'].value_counts())

Regional GPS Data Generated
Total Lots: 50

Region Distribution:
region
Limmu          16
Sidama         13
Guji           11
Yirgacheffe    10
Name: count, dtype: int64


In [8]:
df.head()

,lot_id,supplier_name,washing_station,farm_group,region,district,latitude,longitude,altitude_m,processing_method,...,deforestation_risk,eudr_compliant,farmer_premium_usd,last_audit_date,compliance_status,quantity_bags,price_per_kg_usd,status,available_quantity_kg,last_updated
0,ETH-2025-SID-001,Guji Highland,Bensa,Sidama Union,Sidama,Shakiso,6.956799,38.487953,1671,Washed,...,Low,False,0.83,2025-09-01,Compliant,35,8.27,Available,2100.0,2026-05-25 11:13:41.085344
1,ETH-2025-GUJ-002,Red Cherry Export,Bensa,Limmu Coop,Guji,Yirga Chefe,5.703860,39.197704,2125,Honey,...,High,True,0.85,2025-09-13,Compliant,87,6.06,Available,5220.0,2026-05-25 11:13:41.085344
2,ETH-2025-LIM-003,Daye Bensa,Uraga,Guji Organic,Limmu,Gedeb,8.042708,36.461176,1739,Washed,...,Low,True,0.78,2025-09-25,Compliant,68,7.99,Available,4080.0,2026-05-25 11:13:41.085344
3,ETH-2025-GUJ-004,Sidama Crown,Daye Bensa,Guji Organic,Guji,Hambela,6.157649,39.381638,1919,Natural,...,Low,True,0.83,2025-10-07,Compliant,138,6.90,Available,8280.0,2026-05-25 11:13:41.085344
4,ETH-2025-YIR-005,Sidama Crown,Hambela,Yirga Chefe Coop,Yirgacheffe,Yirga Chefe,6.258898,38.169550,1812,Natural,...,Low,True,0.85,2025-10-19,Compliant,176,6.18,Available,10560.0,2026-05-25 11:13:41.085344


#### Save the Data

In [9]:
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
data_folder = PROJECT_ROOT / "data"
processed_folder = data_folder / "processed"
processed_folder.mkdir(parents=True, exist_ok=True)

df.to_csv(data_folder / "mock_traceability_data.csv", index=False)
df.to_pickle(processed_folder / "traceability_data.pkl")
df.to_excel(processed_folder / "traceability_data.xlsx", index=False)

print("Data saved successfully!")

Data saved successfully!


#### Summary

In [10]:
print("=== COMPLETE ===\n")
print(f"Total Lots Generated        : {len(df)}")
print(f"Regions Covered             : {df['region'].nunique()}")
print(f"Average SCA Score           : {df['sca_score'].mean():.2f}")
print(f"EUDR Compliant Rate         : {df['eudr_compliant'].mean()*100:.1f}%")
print(f"Low Deforestation Risk      : {(df['deforestation_risk'] == 'Low').sum()}")

=== COMPLETE ===

Total Lots Generated        : 50
Regions Covered             : 4
Average SCA Score           : 88.16
EUDR Compliant Rate         : 90.0%
Low Deforestation Risk      : 39
